# Chapitre 5 — Nettoyer : valeurs manquantes, doublons, texte, types, dates

## Le problème

Le fichier `commandes_brutes.csv` est un export réaliste : prix saisis comme du texte avec une virgule et un symbole €, canaux écrits de quatre façons différentes, dates au format belge, quantités manquantes ou écrites en lettres, lignes en double.

Aucun calcul fiable n'est possible sur ces données en l'état. Le nettoyage n'est pas une étape annexe : dans un projet d'IA, c'est souvent la partie la plus longue, et celle qui influence le plus la qualité du modèle.

Ce chapitre suit l'ordre d'un nettoyage réel :

1. diagnostiquer ;
2. traiter les doublons ;
3. traiter les valeurs manquantes ;
4. nettoyer le texte ;
5. convertir les types ;
6. traiter les dates ;
7. modifier un DataFrame sans piège (*Copy-on-Write*).

In [ ]:
import numpy as np
import pandas as pd

brut = pd.read_csv("../donnees/demo/commandes_brutes.csv", sep=";")
brut.head()

## 1. Diagnostiquer

In [ ]:
brut.info()

Constats :

- `quantite` est du texte (`str`) alors qu'on attend des entiers : au moins une valeur n'est pas un nombre ;
- `prix_unitaire` et `date_commande` sont du texte ;
- plusieurs colonnes ont des valeurs manquantes.

In [ ]:
brut["canal"].unique()

In [ ]:
brut["quantite"].unique()

## 2. Les doublons

`duplicated()` renvoie un masque : `True` pour chaque ligne identique à une ligne **précédente**.

In [ ]:
brut.duplicated().sum()

In [ ]:
brut[brut.duplicated(keep=False)].sort_values("commande_id")     # keep=False : toutes les occurrences

`drop_duplicates()` les supprime. Le paramètre `subset` restreint la comparaison à certaines colonnes : ici, un identifiant de commande ne devrait apparaître qu'une fois.

In [ ]:
propre = brut.drop_duplicates(subset="commande_id", keep="first")
propre.shape

> **Pourquoi ?** Dans l'exercice JO, une équipe de football médaillée d'or compte onze lignes dans le fichier. Combien de médailles doit-on lui attribuer ? Quelles colonnes faudrait-il passer à `subset` pour les compter avec `drop_duplicates` ? Le chapitre 7 y revient.

## 3. Les valeurs manquantes

### Repérer

Pandas représente une valeur manquante par `NaN` pour les nombres et le texte, `NaT` pour les dates, `<NA>` pour les types « nullables » (`Int64`, `boolean`…). `isna()` les repère toutes ; `notna()` est son contraire. (`isnull` et `notnull` sont des alias.)

In [ ]:
propre.isna().sum()

In [ ]:
propre[propre["statut"].isna()]

### Supprimer : `dropna`

| Appel | Effet |
|---|---|
| `df.dropna()` | supprime toute ligne contenant au moins un `NaN` |
| `df.dropna(subset=["col"])` | seulement si `col` est manquante |
| `df.dropna(how="all")` | seulement si toute la ligne est vide |
| `df.dropna(axis=1)` | supprime les colonnes contenant un `NaN` |

In [ ]:
print(len(propre), "→", len(propre.dropna()), "lignes avec dropna()")
print(len(propre), "→", len(propre.dropna(subset=["statut"])), "lignes avec dropna(subset=['statut'])")

Supprimer est simple, mais chaque ligne supprimée est une information perdue. On ne supprime que ce qu'on ne peut pas reconstituer, et on mesure combien de lignes on perd.

### Remplacer : `fillna`

In [ ]:
propre["statut"].fillna("inconnu").value_counts()

`fillna` accepte un dictionnaire pour traiter chaque colonne différemment :

In [ ]:
propre.fillna({"statut": "inconnu", "prix_unitaire": "0,00 €"}).isna().sum()

### Imputer une valeur numérique

Pour une variable numérique, on remplace souvent par une statistique : la moyenne ou la médiane.

In [ ]:
ages = pd.read_csv("../donnees/demo/clients.csv")["age"]
print("moyenne :", round(ages.mean(), 1), "| médiane :", ages.median())
ages.fillna(ages.median()).isna().sum()

> **Comparaison.** Sur des revenus `[1800, 2100, 2300, 2500, 45000, NaN]`, quelle valeur de remplacement est la plus représentative : la moyenne ou la médiane ? Calculez les deux.

In [ ]:
revenus = pd.Series([1800, 2100, 2300, 2500, 45000, np.nan])
revenus.mean(), revenus.median()

La moyenne est tirée vers le haut par la valeur extrême ; la médiane ne l'est pas. Sur une distribution asymétrique, la médiane est un choix plus robuste.

**En machine learning**, la statistique d'imputation se calcule uniquement sur les données d'entraînement, puis s'applique telle quelle aux données de test. La calculer sur tout le jeu de données laisserait de l'information du test « fuir » vers l'entraînement (*data leakage*).

Sur un DataFrame aux colonnes mixtes, `median()` doit ignorer le texte : on le précise avec `numeric_only=True`.

In [ ]:
clients = pd.read_csv("../donnees/demo/clients.csv")
clients.fillna(clients.median(numeric_only=True)).isna().sum()

## 4. Nettoyer le texte : l'accesseur `.str`

Les méthodes de chaîne de Python s'appliquent à toute une colonne via `.str`. Les valeurs manquantes sont ignorées.

In [ ]:
propre["canal"].str.strip().str.lower().unique()

Les principales méthodes :

| Méthode | Effet |
|---|---|
| `str.strip()`, `str.lower()`, `str.upper()`, `str.capitalize()` | espaces, casse |
| `str.replace(motif, remplacement)` | remplacer (motif littéral par défaut, regex avec `regex=True`) |
| `str.contains(motif)` | masque booléen (regex par défaut) |
| `str.startswith()`, `str.endswith()` | masque booléen |
| `str.split(sep)` | liste de morceaux |
| `str[i]`, `str[a:b]` | caractère ou sous-chaîne |
| `str.find(sous_chaine)` | position, -1 si absent |
| `str.len()` | longueur |
| `str.cat()` ou `+` | concaténer |

In [ ]:
noms = pd.Series(["  Amina EL idrissi ", "lucas-martin", "SOFIA", None])
noms.str.strip().str.title()

In [ ]:
noms.str.split("-")

In [ ]:
noms.str.strip().str[0]          # première lettre

### Les expressions régulières

`str.contains`, `str.replace(..., regex=True)`, `str.extract` et `str.split(..., regex=True)` acceptent des expressions régulières (*regex*). Quelques motifs suffisent pour la plupart des besoins :

| Motif | Signification |
|---|---|
| `Owens\|Jesse` | « Owens » ou « Jesse » |
| `^A` | commence par A |
| `ez$` | se termine par « ez » |
| `^..E` | troisième caractère E |
| `[0-9]+` | un ou plusieurs chiffres |
| `\s` | un espace |
| `.` | n'importe quel caractère (`\.` pour un point littéral) |

In [ ]:
jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")
jo[jo["Name"].str.contains("Owens|Jesse")]["Name"].nunique()

> **Comparaison.** Le résultat change-t-il avec `case=False` ? Pourquoi ?

In [ ]:
jo[jo["Name"].str.contains("Owens|Jesse", case=False)]["Name"].nunique()

Sans `case=False`, la recherche distingue majuscules et minuscules : un nom contenant « jesse » en minuscules (par exemple au milieu d'un mot) n'est pas retenu.

In [ ]:
actor = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name="actor")
actor[actor["last_name"].str.contains("^..E")][["first_name", "last_name"]].head()

Les motifs regex s'écrivent dans des chaînes brutes `r"..."` dès qu'ils contiennent une barre oblique inverse, sinon Python affiche un `SyntaxWarning` : `r"[-\(\)]"` et non `"[-\(\)]"`.

`str.extract` capture une partie du texte dans une nouvelle colonne :

In [ ]:
jo["Event"].str.extract(r"(Men's|Women's|Mixed)")[0].value_counts(dropna=False)

## 5. Convertir les types

### Du texte vers un nombre

Le prix est écrit `"79,00 €"`. Il faut retirer le symbole et les espaces, remplacer la virgule par un point, puis convertir.

In [ ]:
prix = (propre["prix_unitaire"]
        .str.replace("€", "")
        .str.strip()
        .str.replace(",", ".")
        .astype(float))
prix.head()

`astype` échoue au premier texte non convertible. Pour la quantité, qui contient `"deux"`, on utilise `pd.to_numeric` avec `errors="coerce"` : les valeurs non convertibles deviennent `NaN` au lieu de lever une erreur.

In [ ]:
propre["quantite"].astype(float)

In [ ]:
quantite = pd.to_numeric(propre["quantite"], errors="coerce")
quantite.isna().sum()

`errors="coerce"` transforme un problème bruyant en problème silencieux : il faut toujours compter les `NaN` créés, et regarder les valeurs d'origine concernées.

In [ ]:
propre.loc[quantite.isna(), "quantite"].unique()

### Vers une catégorie

Une colonne qui ne prend que quelques valeurs se convertit en `category`. Sur le fichier JO, l'économie est importante.

In [ ]:
avant = jo.memory_usage(deep=True).sum() / 1e6
jo_cat = jo.astype({c: "category" for c in ["Sex", "NOC", "Games", "Season", "City", "Sport", "Medal"]})
apres = jo_cat.memory_usage(deep=True).sum() / 1e6
print(f"{avant:.1f} Mo → {apres:.1f} Mo")

Le type `category` a deux contreparties : on ne peut affecter qu'une valeur faisant partie des catégories déclarées, et les opérations de texte (`.str`) sont moins directes. On l'utilise pour des colonnes stables et répétitives.

### Remplacer des valeurs

`replace` substitue des valeurs exactes, avec un dictionnaire :

In [ ]:
propre["statut"].replace({"livrée": "livree", "annulée": "annulee"}).unique()

## 6. Les dates

### Convertir

`pd.to_datetime` transforme du texte en dates. Préciser le format évite toute ambiguïté : `02/01/2025` est le 2 janvier en Belgique, le 1er février aux États-Unis.

In [ ]:
dates = pd.to_datetime(propre["date_commande"], format="%d/%m/%Y %H:%M")
dates.head(3)

Codes de format courants : `%Y` année sur 4 chiffres, `%m` mois, `%d` jour, `%H` heure, `%M` minutes, `%S` secondes. Une date au format ISO (`2025-01-02 20:20:00`) se convertit sans préciser de format.

### Extraire des composantes : l'accesseur `.dt`

In [ ]:
pd.DataFrame({
    "date": dates.head(4),
    "annee": dates.head(4).dt.year,
    "mois": dates.head(4).dt.month,
    "nom_mois": dates.head(4).dt.month_name(),
    "jour_semaine": dates.head(4).dt.day_name(),
    "heure": dates.head(4).dt.hour,
})

Ces composantes sont des variables très utiles en machine learning : le jour de la semaine ou l'heure expliquent souvent une partie des comportements d'achat.

### Formater, comparer, calculer

In [ ]:
dates.head(3).dt.strftime("%d/%m/%Y")

In [ ]:
(dates >= pd.Timestamp("2025-12-01")).sum()          # commandes de décembre

In [ ]:
delai = pd.Timestamp("2026-01-01") - dates
delai.head(3)                                          # Timedelta : une durée

In [ ]:
delai.dt.days.describe()

### Créer des dates

In [ ]:
pd.Timestamp.today().normalize(), pd.date_range("2025-01-01", periods=4, freq="W-MON")

`to_datetime` sait aussi relire une date écrite dans un format inhabituel, si on lui donne le format exact :

In [ ]:
texte = pd.Series(pd.date_range("2023-01-01", periods=3)).dt.strftime("[%Y]**(%m)--{%d}")
texte.tolist(), pd.to_datetime(texte, format="[%Y]**(%m)--{%d}").tolist()

## 7. Renommer et supprimer des colonnes

In [ ]:
propre.rename(columns={"prix_unitaire": "prix", "quantite": "qte"}).columns.tolist()

In [ ]:
propre.drop(columns=["canal", "statut"]).columns.tolist()

`del df["col"]` supprime aussi une colonne, directement dans le DataFrame (on le rencontre dans les exercices drills).

## 8. Modifier un DataFrame : Copy-on-Write

En Pandas 3, chaque sélection se comporte comme **une copie indépendante**. Conséquence : une modification doit toujours s'appliquer à l'objet que l'on veut modifier, en **une seule opération**.

> **Pourquoi ?** Exécutez les deux cellules suivantes. La colonne `statut` a-t-elle été modifiée ?

In [ ]:
essai = propre.copy()
essai["statut"].fillna("inconnu", inplace=True)
essai["statut"].isna().sum()

In [ ]:
essai[essai["statut"].isna()]["statut"] = "inconnu"
essai["statut"].isna().sum()

Aucune des deux tentatives ne modifie `essai`. Dans les deux cas, on modifie un **objet intermédiaire** :

- `essai["statut"]` renvoie une Series, que `fillna(..., inplace=True)` modifie, puis qui disparaît ;
- `essai[masque]` renvoie un nouveau DataFrame, dans lequel on écrit, puis qui disparaît.

Pandas l'indique par un avertissement `ChainedAssignmentError`. Les deux écritures correctes :

In [ ]:
essai["statut"] = essai["statut"].fillna("inconnu")          # réaffecter le résultat
essai.loc[essai["canal"].isna(), "canal"] = "inconnu"        # loc[lignes, colonne] = valeur
essai["statut"].isna().sum()

Le paramètre `inplace=True` est d'ailleurs en cours d'abandon (Pandas 3.1 l'annonce comme déprécié sur `drop`, `rename`, `set_index`, `reset_index`…). La forme recommandée est toujours la réaffectation : `df = df.drop(columns=...)`.

Quand on crée un sous-ensemble destiné à être modifié, `.copy()` rend l'intention explicite :

In [ ]:
web = propre[propre["canal"].str.strip().str.lower() == "web"].copy()
web["canal"] = "web"
web.shape

## 9. Le nettoyage complet, enchaîné

Toutes les étapes peuvent s'écrire en une seule chaîne de méthodes. `assign` crée ou remplace des colonnes ; chaque étape reçoit le résultat de la précédente.

In [ ]:
commandes = (
    pd.read_csv("../donnees/demo/commandes_brutes.csv", sep=";")
    .drop_duplicates(subset="commande_id")
    .assign(
        date_commande=lambda d: pd.to_datetime(d["date_commande"], format="%d/%m/%Y %H:%M"),
        quantite=lambda d: pd.to_numeric(d["quantite"], errors="coerce").astype("Int64"),
        prix_unitaire=lambda d: d["prix_unitaire"].str.replace("€", "").str.strip()
                                                  .str.replace(",", ".").astype(float),
        canal=lambda d: d["canal"].str.strip().str.lower().astype("category"),
        statut=lambda d: d["statut"].fillna("inconnu"),
    )
)
commandes.info()

Le type `Int64` (avec une majuscule) est un entier qui accepte les valeurs manquantes `<NA>` : la quantité reste entière malgré ses trous.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| `df["col"].fillna(..., inplace=True)` | aucune modification, avertissement | `df["col"] = df["col"].fillna(...)` |
| `df[masque]["col"] = valeur` | aucune modification | `df.loc[masque, "col"] = valeur` |
| `astype(float)` sur du texte sale | `ValueError` | nettoyer avec `.str`, ou `pd.to_numeric(errors="coerce")` |
| `errors="coerce"` sans contrôle | des `NaN` créés en silence | compter et inspecter les valeurs converties en `NaN` |
| Date sans format | jour et mois inversés | `format="%d/%m/%Y"` |
| Imputer avec la moyenne d'une variable asymétrique | valeur peu représentative | la médiane |
| Motif regex sans `r"..."` | `SyntaxWarning` | chaîne brute |

## Exercices

- Exercice Bstorm **JO**, questions **17 à 22**.
- Exercice Bstorm **Sakila**, questions **1 à 7** (`exercices/bstorm/01.5_pandas_sakila/`).
- Exercice supplémentaire **S5** : les modifications qui ne modifient rien.
- Renforcement : drill **D01 Chipotle** (étapes 13 à 17), **D09 Iris**, puis **D09 Wine** pour approfondir. Ces deux derniers drills utilisent des écritures chaînées : leur version d'origine ne fonctionne plus en Pandas 3 (voir la traduction française, qui le signale).

## À retenir

- Diagnostiquer d'abord : `info()`, `unique()`, `isna().sum()`, `duplicated().sum()`.
- `drop_duplicates(subset=...)`, `dropna(subset=...)`, `fillna(valeur ou dict)`.
- `.str` pour le texte, avec les regex ; `.dt` pour les dates.
- `pd.to_numeric(errors="coerce")` et `pd.to_datetime(format=...)` pour convertir.
- `category` pour les colonnes répétitives.
- Une modification passe par `df["col"] = ...` ou `df.loc[...] = ...`, jamais par un objet intermédiaire.